# Convolutional Neural Networks (CNNs) with PyTorch

This notebook covers CNNs and trains one on the MNIST digit classification task.

**Topics covered:**
- What is Convolution?
- Filters, Stride, Padding
- Pooling & Normalization
- Full CNN architecture
- Training & Evaluation on MNIST


## 1. What is Convolution?

A **convolution** slides a small matrix (called a **filter** or **kernel**) across an image and computes dot products at each position. Each filter detects a specific feature (edges, curves, textures…).

Key parameters:
| Parameter | Effect |
|-----------|--------|
| **Kernel size** | Size of the sliding filter (e.g. 3×3) |
| **Stride** | How many pixels to skip per step |
| **Padding** | Zeros added around the border to control output size |

**Why CNNs for images?**  
Images are grid-structured data. CNNs exploit this by sharing weights spatially — the same filter scans the entire image, making them much more efficient than fully connected layers.


## 2. Pooling & Normalization

**MaxPooling** — downsamples the feature map by taking the max value in each window. Reduces size and adds translation invariance.

**Batch Normalization** — normalizes activations within each mini-batch. Speeds up training and reduces sensitivity to weight initialization.


## 3. CNN Architecture

```
Input (1×28×28)
    ↓
Conv1 (32 filters, 3×3) → ReLU → MaxPool
    ↓ 32×13×13
Conv2 (64 filters, 3×3) → ReLU → MaxPool
    ↓ 64×5×5
Flatten → FC(128) → Dropout → FC(10)
    ↓
Output (10 classes)
```


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
NUM_EPOCHS    = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Data ──────────────────────────────────────────────────────────────
# Normalize using MNIST pre-computed mean and std
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")


In [ ]:
# ── Model ─────────────────────────────────────────────────────────────
class SmallCNN(nn.Module):
    """
    Two convolutional blocks followed by a fully-connected classifier.

    Input  → 1×28×28
    Conv1  → 32×26×26  (3×3 kernel, no padding)
    Pool1  → 32×13×13  (2×2 max-pool)
    Conv2  → 64×11×11  (3×3 kernel)
    Pool2  → 64×5×5    (2×2 max-pool)
    Flatten→ 1600
    FC1    → 128
    FC2    → 10 (one score per digit)
    """

    def __init__(self):
        super(SmallCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        # Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        # Classifier
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(64 * 5 * 5, 128)
        self.relu3   = nn.ReLU()
        self.dropout = nn.Dropout(p=0.5)   # reduces overfitting
        self.fc2     = nn.Linear(128, 10)  # 10 digit classes

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))   # → (batch, 32, 13, 13)
        x = self.pool2(self.relu2(self.conv2(x)))   # → (batch, 64,  5,  5)
        x = self.flatten(x)                          # → (batch, 1600)
        x = self.relu3(self.fc1(x))                 # → (batch, 128)
        x = self.dropout(x)
        x = self.fc2(x)                             # → (batch, 10) logits
        return x


model     = SmallCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print(f"\nTrainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# ── Training & Evaluation Loops ───────────────────────────────────────
def train(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss, correct = 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        outputs = model(images)
        loss    = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()

        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} | Step {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / len(loader.dataset)
    print(f"→ Train Epoch {epoch}: Loss={avg_loss:.4f}, Acc={accuracy:.2f}%")


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            correct    += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / len(loader.dataset)
    print(f"→ Test            : Loss={avg_loss:.4f}, Acc={accuracy:.2f}%\n")
    return accuracy


# ── Run ───────────────────────────────────────────────────────────────
best_accuracy = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train(model, train_loader, criterion, optimizer, epoch)
    acc = evaluate(model, test_loader, criterion)

    if acc > best_accuracy:
        best_accuracy = acc
        torch.save(model.state_dict(), "best_mnist_cnn.pth")
        print(f"  ✓ New best model saved (acc={acc:.2f}%)\n")

print(f"Training complete. Best test accuracy: {best_accuracy:.2f}%")


In [ ]:
# ── Quick Inference Demo ──────────────────────────────────────────────
import matplotlib.pyplot as plt

model.load_state_dict(torch.load("best_mnist_cnn.pth", map_location=DEVICE))
model.eval()

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    img, label = test_dataset[i]
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(DEVICE))
        pred   = logits.argmax(dim=1).item()
    ax.imshow(img.squeeze(), cmap="gray")
    color = "green" if pred == label else "red"
    ax.set_title(f"P:{pred}\nT:{label}", color=color, fontsize=8)
    ax.axis("off")

plt.suptitle("CNN Predictions (green=correct, red=wrong)", y=1.05)
plt.tight_layout()
plt.show()
